In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np

# --------------------------------------------------
# Load the current master dataset
# --------------------------------------------------

PROJECT_DIR = Path(
    r"C:\Users\saray\OneDrive\Desktop\Retail"
)

DATA_PATH = (
    PROJECT_DIR
    / "data"
    / "processed"
    / "retail_sales_final.csv"
)

df = pd.read_csv(
    DATA_PATH,
    low_memory=False,
    dtype={"StockCode": "string"}
)

# Parse dates and numeric fields for analysis
df["InvoiceDate"] = pd.to_datetime(
    df["InvoiceDate"],
    format="%Y-%m-%d",
    errors="coerce"
)

for col in ["Quantity", "Price", "Revenue"]:
    df[col] = pd.to_numeric(
        df[col],
        errors="coerce"
    )

# --------------------------------------------------
# Prepare a customer-analysis copy
# Do not modify the master dataset
# --------------------------------------------------

customer_df = df.loc[
    df["Customer ID"].notna()
].copy()

print("=" * 60)
print("RETAILPULSE - CUSTOMER SEGMENTATION DATA CHECK")
print("=" * 60)

print("\nMaster dataset rows:", f"{len(df):,}")
print("Rows with known Customer ID:", f"{len(customer_df):,}")
print(
    "Rows without Customer ID:",
    f"{df['Customer ID'].isna().sum():,}"
)

print(
    "Unique customers:",
    f"{customer_df['Customer ID'].nunique():,}"
)

print(
    "Unique invoices for known customers:",
    f"{customer_df['Invoice'].nunique():,}"
)

print(
    "Date range:",
    customer_df["InvoiceDate"].min(),
    "to",
    customer_df["InvoiceDate"].max()
)

print(
    "Rows with negative quantities:",
    f"{(customer_df['Quantity'] < 0).sum():,}"
)

print(
    "Rows with zero prices:",
    f"{(customer_df['Price'] == 0).sum():,}"
)

print("\nData loaded. Master CSV was not modified.")

In [ ]:
import pandas as pd
from pathlib import Path

# ============================================================
# RETAILPULSE - RFM CUSTOMER FEATURE ENGINEERING
# ============================================================

PROJECT_DIR = Path(
    r"C:\Users\saray\OneDrive\Desktop\Retail"
)

PROCESSED_DIR = PROJECT_DIR / "data" / "processed"
REPORT_DIR = PROJECT_DIR / "reports" / "segmentation"

REPORT_DIR.mkdir(parents=True, exist_ok=True)

# Work from the loaded customer dataset.
# Do not modify the master CSV.
seg_input = customer_df.copy()

# Keep valid purchase transactions for RFM analysis
purchase_data = seg_input[
    seg_input["Customer ID"].notna()
    & seg_input["InvoiceDate"].notna()
    & (seg_input["Quantity"] > 0)
    & (seg_input["Price"] > 0)
].copy()

if purchase_data.empty:
    raise ValueError("No eligible purchase transactions found.")

# Calculate monetary value consistently from quantity × price
purchase_data["PurchaseValue"] = (
    purchase_data["Quantity"] * purchase_data["Price"]
)

# Recency reference date: one day after the latest transaction date
snapshot_date = (
    purchase_data["InvoiceDate"].max().normalize()
    + pd.Timedelta(days=1)
)

# Aggregate transaction rows to one row per customer
rfm = (
    purchase_data
    .groupby("Customer ID")
    .agg(
        LastPurchase=("InvoiceDate", "max"),
        Frequency=("Invoice", "nunique"),
        Monetary=("PurchaseValue", "sum"),
        TotalQuantity=("Quantity", "sum"),
        UniqueProducts=("StockCode", "nunique")
    )
    .reset_index()
)

# Recency is the number of days since the latest purchase
rfm["Recency"] = (
    snapshot_date - rfm["LastPurchase"].dt.normalize()
).dt.days

# Create quartile-based RFM scores.
# Rank first to handle tied values without qcut errors.
rfm["R_Score"] = pd.qcut(
    rfm["Recency"].rank(method="first"),
    q=4,
    labels=[4, 3, 2, 1]
).astype(int)

rfm["F_Score"] = pd.qcut(
    rfm["Frequency"].rank(method="first"),
    q=4,
    labels=[1, 2, 3, 4]
).astype(int)

rfm["M_Score"] = pd.qcut(
    rfm["Monetary"].rank(method="first"),
    q=4,
    labels=[1, 2, 3, 4]
).astype(int)

rfm["RFM_Score"] = (
    rfm["R_Score"].astype(str)
    + rfm["F_Score"].astype(str)
    + rfm["M_Score"].astype(str)
)

# Save as a separate feature file; don't overwrite existing outputs
rfm_path = PROCESSED_DIR / "customer_segmentation_rfm_rebuilt.csv"

rfm.to_csv(rfm_path, index=False)

# Save a compact summary for the report
summary = pd.DataFrame({
    "Metric": [
        "Eligible purchase rows",
        "Customers in RFM table",
        "Unique invoices in eligible data",
        "RFM snapshot date",
        "Median recency (days)",
        "Median frequency (invoices)",
        "Median monetary value"
    ],
    "Value": [
        len(purchase_data),
        rfm["Customer ID"].nunique(),
        purchase_data["Invoice"].nunique(),
        snapshot_date.date(),
        rfm["Recency"].median(),
        rfm["Frequency"].median(),
        round(rfm["Monetary"].median(), 2)
    ]
})

summary.to_csv(
    REPORT_DIR / "rfm_feature_summary.csv",
    index=False
)

print("=" * 60)
print("RETAILPULSE - RFM FEATURE ENGINEERING")
print("=" * 60)

print("\nEligible purchase rows:", f"{len(purchase_data):,}")
print("Customers in RFM table:", f"{len(rfm):,}")
print("Snapshot date:", snapshot_date.date())

print("\nRFM summary:")
print(
    rfm[[
        "Recency", "Frequency", "Monetary",
        "TotalQuantity", "UniqueProducts"
    ]].describe().round(2).to_string()
)

print("\nRFM score distribution:")
print(rfm["RFM_Score"].value_counts().head(15).to_string())

print("\nFirst 10 customers:")
print(rfm.head(10).to_string(index=False))

print("\nRFM file saved to:", rfm_path)
print("Summary saved to:", REPORT_DIR / "rfm_feature_summary.csv")
print("\nMaster dataset was not modified.")

In [ ]:
import pandas as pd
import numpy as np

from pathlib import Path
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans, DBSCAN
from sklearn.metrics import silhouette_score

# ============================================================
# RETAILPULSE - CUSTOMER CLUSTERING BENCHMARK
# ============================================================

PROJECT_DIR = Path(
    r"C:\Users\saray\OneDrive\Desktop\Retail"
)

PROCESSED_DIR = PROJECT_DIR / "data" / "processed"
REPORT_DIR = PROJECT_DIR / "reports" / "segmentation"

PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
REPORT_DIR.mkdir(parents=True, exist_ok=True)

# Use the RFM table created in the previous cell
features = [
    "Recency",
    "Frequency",
    "Monetary",
    "TotalQuantity",
    "UniqueProducts"
]

cluster_data = rfm[
    ["Customer ID"] + features
].copy()

# Ensure valid numeric input
cluster_data[features] = cluster_data[features].apply(
    pd.to_numeric, errors="coerce"
)

cluster_data = cluster_data.replace(
    [np.inf, -np.inf], np.nan
).dropna(subset=features)

if len(cluster_data) < 100:
    raise ValueError(
        "Too few valid customers for clustering."
    )

# Log-transform right-skewed features
X_log = np.log1p(cluster_data[features].clip(lower=0))

# Standardize features so large-scale variables do not dominate
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_log)

print("=" * 65)
print("RETAILPULSE - K-MEANS AND DBSCAN BENCHMARK")
print("=" * 65)

print("\nCustomers used:", f"{len(cluster_data):,}")
print("Features:", features)

# ------------------------------------------------------------
# 1. K-MEANS: TEST 6, 7, AND 8 CLUSTERS
# ------------------------------------------------------------

kmeans_results = []
kmeans_models = {}

for k in [6, 7, 8]:
    model = KMeans(
        n_clusters=k,
        random_state=42,
        n_init=20
    )

    labels = model.fit_predict(X_scaled)

    # Use a reproducible sample to calculate silhouette
    sample_size = min(2000, len(X_scaled))

    score = silhouette_score(
        X_scaled,
        labels,
        sample_size=sample_size,
        random_state=42
    )

    kmeans_models[k] = (model, labels)

    kmeans_results.append({
        "Model": "KMeans",
        "Clusters": k,
        "Silhouette_Score": score,
        "Inertia": model.inertia_,
        "Customers": len(labels)
    })

kmeans_scores = pd.DataFrame(kmeans_results)

print("\nK-MEANS COMPARISON")
print(kmeans_scores.round(4).to_string(index=False))

# Choose the highest silhouette score among the tested k values
best_k = int(
    kmeans_scores.loc[
        kmeans_scores["Silhouette_Score"].idxmax(),
        "Clusters"
    ]
)

best_model, best_labels = kmeans_models[best_k]

print(f"\nBest tested K-Means configuration: k={best_k}")

# Save customer cluster assignments
kmeans_output = cluster_data.copy()
kmeans_output["KMeans_Cluster"] = best_labels

kmeans_path = (
    PROCESSED_DIR / "customer_segmentation_kmeans_rebuilt.csv"
)

kmeans_output.to_csv(kmeans_path, index=False)

# Save cluster profiles for business interpretation
cluster_profiles = (
    kmeans_output
    .groupby("KMeans_Cluster")
    .agg(
        Customers=("Customer ID", "count"),
        Median_Recency=("Recency", "median"),
        Median_Frequency=("Frequency", "median"),
        Median_Monetary=("Monetary", "median"),
        Median_Quantity=("TotalQuantity", "median"),
        Median_Unique_Products=("UniqueProducts", "median")
    )
    .reset_index()
)

cluster_profiles.to_csv(
    REPORT_DIR / "kmeans_cluster_profiles.csv",
    index=False
)

# ------------------------------------------------------------
# 2. DBSCAN: TEST SEVERAL DENSITY SETTINGS
# ------------------------------------------------------------

dbscan_results = []
best_dbscan = None
best_dbscan_score = -1

for eps in [0.5, 0.75, 1.0, 1.25, 1.5]:
    for min_samples in [5, 10, 15]:

        model = DBSCAN(
            eps=eps,
            min_samples=min_samples
        )

        labels = model.fit_predict(X_scaled)

        cluster_labels = set(labels) - {-1}
        n_clusters = len(cluster_labels)
        noise_fraction = float(np.mean(labels == -1))

        # Silhouette is calculated only for non-noise points
        score = np.nan
        non_noise = labels != -1
        core_labels = labels[non_noise]
        core_features = X_scaled[non_noise]

        if (
            n_clusters >= 2
            and len(core_labels) > n_clusters
        ):
            try:
                score = silhouette_score(
                    core_features,
                    core_labels,
                    sample_size=min(2000, len(core_labels)),
                    random_state=42
                )
            except ValueError:
                score = np.nan

        dbscan_results.append({
            "Model": "DBSCAN",
            "Eps": eps,
            "Min_Samples": min_samples,
            "Clusters_Excluding_Noise": n_clusters,
            "Noise_Percentage": noise_fraction * 100,
            "Silhouette_Score": score
        })

        if (
            np.isfinite(score)
            and score > best_dbscan_score
        ):
            best_dbscan_score = score
            best_dbscan = {
                "eps": eps,
                "min_samples": min_samples,
                "labels": labels
            }

dbscan_scores = pd.DataFrame(dbscan_results)

print("\nDBSCAN COMPARISON")
print(dbscan_scores.round(4).to_string(index=False))

# Save benchmark results
kmeans_scores.to_csv(
    REPORT_DIR / "kmeans_benchmark.csv",
    index=False
)

dbscan_scores.to_csv(
    REPORT_DIR / "dbscan_benchmark.csv",
    index=False
)

# Save best valid DBSCAN assignment for inspection
if best_dbscan is not None:
    dbscan_output = cluster_data.copy()
    dbscan_output["DBSCAN_Cluster"] = best_dbscan["labels"]

    dbscan_path = (
        PROCESSED_DIR / "customer_segmentation_dbscan_rebuilt.csv"
    )

    dbscan_output.to_csv(dbscan_path, index=False)

    print(
        "\nBest DBSCAN parameters:",
        f"eps={best_dbscan['eps']},",
        f"min_samples={best_dbscan['min_samples']}"
    )

    print("DBSCAN output:", dbscan_path)
else:
    print(
        "\nDBSCAN did not produce a valid multi-cluster "
        "silhouette result in the tested configurations."
    )

print("\nK-Means output:", kmeans_path)
print(
    "K-Means profiles:",
    REPORT_DIR / "kmeans_cluster_profiles.csv"
)
print("Benchmark reports saved to:", REPORT_DIR)

print("\nMaster dataset was not modified.")

In [ ]:
import pandas as pd
from pathlib import Path

# Project paths
PROJECT_DIR = Path(
    r"C:\Users\saray\OneDrive\Desktop\Retail"
)

PROCESSED_DIR = PROJECT_DIR / "data" / "processed"
REPORT_DIR = PROJECT_DIR / "reports" / "segmentation"

# Load the rebuilt K-Means customer assignments
kmeans_path = (
    PROCESSED_DIR / "customer_segmentation_kmeans_rebuilt.csv"
)

seg = pd.read_csv(kmeans_path)

# Ensure the expected columns exist
required = [
    "Customer ID", "Recency", "Frequency",
    "Monetary", "TotalQuantity",
    "UniqueProducts", "KMeans_Cluster"
]

missing = [col for col in required if col not in seg.columns]

if missing:
    raise ValueError(f"Missing columns in K-Means output: {missing}")

# Summarize cluster sizes and customer behaviour
profiles = (
    seg.groupby("KMeans_Cluster")
    .agg(
        Customers=("Customer ID", "nunique"),
        Median_Recency_Days=("Recency", "median"),
        Median_Invoice_Frequency=("Frequency", "median"),
        Median_Monetary_Value=("Monetary", "median"),
        Median_Total_Quantity=("TotalQuantity", "median"),
        Median_Unique_Products=("UniqueProducts", "median")
    )
    .reset_index()
)

profiles["Customer_Share_Percent"] = (
    profiles["Customers"] / seg["Customer ID"].nunique() * 100
).round(2)

profiles = profiles.sort_values("KMeans_Cluster")

print("=" * 65)
print("RETAILPULSE - K-MEANS CLUSTER PROFILES")
print("=" * 65)

print("\nNumber of clusters:", profiles["KMeans_Cluster"].nunique())
print("Customers assigned:", f"{seg['Customer ID'].nunique():,}")

print("\nCluster sizes:")
print(
    profiles[
        ["KMeans_Cluster", "Customers", "Customer_Share_Percent"]
    ].to_string(index=False)
)

print("\nBehavioural profiles:")
print(profiles.round(2).to_string(index=False))

# Flag very small clusters for review
small_clusters = profiles[
    profiles["Customer_Share_Percent"] < 1
]

print("\nClusters containing less than 1% of customers:")
if small_clusters.empty:
    print("None")
else:
    print(
        small_clusters[
            ["KMeans_Cluster", "Customers", "Customer_Share_Percent"]
        ].to_string(index=False)
    )

# Save the profile report
profile_path = REPORT_DIR / "kmeans_cluster_profiles_rebuilt.csv"
profiles.to_csv(profile_path, index=False)

print("\nProfiles saved to:", profile_path)
print("Master dataset was not modified.")

In [ ]:
import pandas as pd
from pathlib import Path

# ============================================================
# RETAILPULSE - BUSINESS INTERPRETATION OF CUSTOMER SEGMENTS
# ============================================================

PROJECT_DIR = Path(
    r"C:\Users\saray\OneDrive\Desktop\Retail"
)

PROCESSED_DIR = PROJECT_DIR / "data" / "processed"
REPORT_DIR = PROJECT_DIR / "reports" / "segmentation"

PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
REPORT_DIR.mkdir(parents=True, exist_ok=True)

# Load the rebuilt K-Means assignments
input_path = (
    PROCESSED_DIR / "customer_segmentation_kmeans_rebuilt.csv"
)

seg = pd.read_csv(input_path)

# Map the observed profiles to initial business labels
segment_labels = {
    0: "Recent Low-Frequency Customers",
    1: "Hibernating Customers",
    2: "Recent Potential Loyalists",
    3: "At-Risk Customers",
    4: "Champions",
    5: "Lost Low-Value Customers",
    6: "Loyal High-Value Customers"
}

segment_actions = {
    0: "Encourage repeat purchases with relevant recommendations.",
    1: "Test a reactivation campaign and monitor responses.",
    2: "Use loyalty incentives and relevant cross-selling.",
    3: "Test a win-back campaign and monitor engagement.",
    4: "Prioritize retention and consider suitable VIP benefits.",
    5: "Test a low-cost reactivation offer before increasing spend.",
    6: "Reward loyalty and recommend relevant complementary products."
}

# Confirm that all expected cluster IDs are present
found_clusters = set(seg["KMeans_Cluster"].unique())
expected_clusters = set(segment_labels.keys())

if found_clusters != expected_clusters:
    raise ValueError(
        f"Unexpected cluster IDs. Found: {sorted(found_clusters)}"
    )

# Add business labels and recommended actions
seg["Segment"] = seg["KMeans_Cluster"].map(segment_labels)
seg["Recommended_Action"] = seg["KMeans_Cluster"].map(segment_actions)

# Build a summary from the saved customer assignments
segment_summary = (
    seg.groupby(
        ["KMeans_Cluster", "Segment", "Recommended_Action"],
        as_index=False
    )
    .agg(
        Customers=("Customer ID", "nunique"),
        Median_Recency_Days=("Recency", "median"),
        Median_Invoice_Frequency=("Frequency", "median"),
        Median_Monetary_Value=("Monetary", "median"),
        Median_Unique_Products=("UniqueProducts", "median")
    )
)

total_customers = seg["Customer ID"].nunique()

segment_summary["Customer_Share_Percent"] = (
    segment_summary["Customers"] / total_customers * 100
).round(2)

segment_summary = segment_summary.sort_values("KMeans_Cluster")

# Save labeled assignments and report summary
labeled_path = (
    PROCESSED_DIR / "customer_segmentation_kmeans_labeled.csv"
)

summary_path = (
    REPORT_DIR / "customer_segment_business_summary.csv"
)

seg.to_csv(labeled_path, index=False)
segment_summary.to_csv(summary_path, index=False)

print("=" * 65)
print("RETAILPULSE - CUSTOMER SEGMENTATION SUMMARY")
print("=" * 65)

print("\nTotal customers:", f"{total_customers:,}")
print("Number of segments:", segment_summary["KMeans_Cluster"].nunique())

print("\nBusiness segment profiles:")
print(
    segment_summary[
        [
            "KMeans_Cluster",
            "Segment",
            "Customers",
            "Customer_Share_Percent",
            "Median_Recency_Days",
            "Median_Invoice_Frequency",
            "Median_Monetary_Value"
        ]
    ].round(2).to_string(index=False)
)

print("\nLabeled customer file:")
print(labeled_path)

print("\nBusiness summary:")
print(summary_path)

print("\nMaster dataset was not modified.")